This script processes each academic paper in Markdown format using the OpenAI API.
It applies a set of 19 pre-written prompts to each full paper in order to extract 
structured information across standardized categories (e.g., research problem, methods, findings).

The results are saved in JSON format as a list of dictionaries, one per paper.

⚠️ Important:
You must create a `.env` file in the same directory with the following line:
OPENAI_API_KEY="your_openai_api_key_here"

In [ ]:
# Import libraries
import os
import openai
import json
import time
from dotenv import load_dotenv

In [ ]:
# Load API key from .env file
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OpenAI API Key is missing in the .env file.")

# Initialize OpenAI client
client = openai.OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
# Base path where extracted paper folders and Markdown files are located
BASE_PATH = "Houston_pdfs/output"

In [ ]:
# Load extraction prompts (one per category)
with open("prompts.json", "r", encoding="utf-8") as f:
    prompts = json.load(f)

def process_paper(paper_id):
    """
    Sends the full content of a paper along with each prompt independently.
    The extracted answers are saved in a dictionary.
    """
    paper_path = os.path.join(BASE_PATH, paper_id, "content.md")
    
    if not os.path.exists(paper_path):
        print(f"⚠️ Skipping {paper_id}: content.md not found.")
        return None

    with open(paper_path, "r", encoding="utf-8") as f:
        paper_content = f.read()

    extracted_info = {"Paper ID": paper_id}

    # Iterate over each prompt and send full content each time
    for key, prompt in prompts.items():
        try:
            full_prompt = (
                "You are an AI assistant that extracts structured data from academic papers.\n\n"
                "Here is a research paper:\n\n"
                "---\n"
                f"{paper_content}\n"
                "---\n\n"
                f"Now, please answer the following:\n{prompt}"
            )

            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": "You are an AI assistant that extracts structured data from academic papers."},
                    {"role": "user", "content": full_prompt}
                ],
                temperature=0.1
            )

            extracted_info[key] = response.choices[0].message.content.strip()

        except Exception as e:
            print(f"❌ Error extracting {key} for {paper_id}: {e}")
            extracted_info[key] = None

        time.sleep(1)  # Respect API rate limits
    
    return extracted_info

In [ ]:
def process_all_papers():
    """
    Processes all paper folders inside BASE_PATH and extracts information for each.
    The result is saved in a JSON file.
    """
    results = []
    for paper_id in os.listdir(BASE_PATH):
        paper_folder = os.path.join(BASE_PATH, paper_id)
        if os.path.isdir(paper_folder):
            print(f"📄 Processing paper: {paper_id}")
            paper_data = process_paper(paper_id)
            if paper_data:
                results.append(paper_data)

    with open("extracted_results_with_context.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    print("✅ Extraction complete. Results saved to extracted_results_with_context.json.")


In [ ]:
process_all_papers()